## A language tutor bot to help speakers improve their chinese!


In [ ]:
# imports

import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import sqlite3
from pathlib import Path
from datetime import datetime as dt


In [ ]:
# Initialization

load_dotenv(override=True)

openai_api_key = os.getenv("OPENAI_API_KEY")
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

MODEL = "gpt-4.1-mini"
openai = OpenAI()

DB = "words.db"


### Change the system prompt to a language tutor


In [ ]:
system_message = """
You are a patient Chinese tutor. Reply in Chinese at HSK4-5 level. After your reply, briefly note any grammar/word-choice mistakes the user made (in English). Suggest 1 better word/phrase per turn.
Always be accurate. If you don't know the answer, say so.
"""
# the last sentence helps to minimise hallucinations.


### We will have 2 tools here - save to review list and get the review list


In [ ]:
def init_db() -> None:
    """Create the table if it doesn't exist. Idempotent — safe to call every run."""
    with sqlite3.connect(DB) as conn:
        conn.execute("""
            CREATE TABLE IF NOT EXISTS words (
                hanzi      TEXT PRIMARY KEY,
                pinyin     TEXT NOT NULL,
                meaning    TEXT NOT NULL,
                example    TEXT,
                saved_at   TEXT NOT NULL
            )
        """)
        # commit happens automatically when the `with` block exits cleanly


init_db()  # call once at module import


In [ ]:
def save_to_review_list(hanzi: str, pinyin: str, meaning: str, example: str = ""):
    """Save a Chinese word/phrase to the user's review list. Idempotent on hanzi."""
    print(f"DATABASE TOOL CALLED: Saving word {hanzi}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute(
            "INSERT OR IGNORE INTO words (hanzi, pinyin, meaning, example, saved_at) VALUES (?, ?, ?, ?, ?)",
            (hanzi, pinyin, meaning, example, dt.now().isoformat()),
        )
        conn.commit()
        # cursor.rowcount = 1 if inserted, 0 if skipped due to existing hanzi
        result = {
            "status": "saved" if cursor.rowcount else "already_saved",
            "hanzi": hanzi,
        }

    conn.close()

    return result


In [ ]:
print(save_to_review_list("学习", "xué xí", "to study", "我喜欢学习中文。"))


In [ ]:
def get_review_list(limit: int = 20) -> dict:
    """Return saved words, most recent first."""
    with sqlite3.connect(DB) as conn:
        rows = conn.execute(
            "SELECT hanzi, pinyin, meaning, example, saved_at "
            "FROM words ORDER BY saved_at DESC LIMIT ?",
            (limit,),
        ).fetchall()
        total = conn.execute("SELECT COUNT(*) FROM words").fetchone()[0]

    conn.close()

    print(rows)
    print(total)

    # tools function MUST return something otherwise the AI will have NO idea what you're talking about.
    return {
        "total_count": total,
        "words": [
            {
                "word": r[0],
                "pinyin": r[1],
                "meaning": r[2],
                "example": r[3],
                "created_at": r[4],
            }
            for r in rows
        ],  # Row → dict so it serializes to JSON
    }

In [ ]:
get_review_list()

### Draft the json for the tools


In [ ]:
save_to_review_list_schema = {
    "type": "function",
    "function": {
        "name": "save_to_review_list",
        "description": (
            "Save a Chinese word or phrase to the user's personal review list "
            "for later quizzing. Call this whenever you introduce a new word, "
            "correct the user's vocabulary mistake, or the user explicitly says "
            "'save this' / '记下来' / '收藏'."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "hanzi": {
                    "type": "string",
                    "description": "The Chinese characters (simplified). E.g. '学习'",
                },
                "pinyin": {
                    "type": "string",
                    "description": "Pinyin with tone numbers or marks. E.g. 'xue2 xi2' or 'xuéxí'",
                },
                "meaning": {
                    "type": "string",
                    "description": "Concise English meaning. E.g. 'to study; to learn'",
                },
                "example": {
                    "type": "string",
                    "description": "Optional example sentence in Chinese using this word.",
                },
            },
            "required": ["hanzi", "pinyin", "meaning"],
            "additionalProperties": False,
        },
    },
}

get_review_list_schema = {
    "type": "function",
    "function": {
        "name": "get_review_list",
        "description": (
            "Retrieve the user's saved review list. Call this when the user asks "
            "to be quizzed, asks 'what have I learned?', requests a review, or "
            "says '复习' / '考我'."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "limit": {
                    "type": "integer",
                    "description": "Max words to return (default 20).",
                    "default": 20,
                },
            },
            "required": [],
            "additionalProperties": False,
        },
    },
}

tools = [save_to_review_list_schema, get_review_list_schema]


### Drafting the chat function and the tool calls function


In [ ]:
def handle_tool_calls(message) -> list[dict]:
    """Dispatch tool calls from the LLM to the actual Python functions."""
    tool_responses = []

    for tool_call in message.tool_calls:
        fn_name = tool_call.function.name
        args = json.loads(tool_call.function.arguments)

        # Dispatch
        if fn_name == "save_to_review_list":
            result = save_to_review_list(**args)
        elif fn_name == "get_review_list":
            result = get_review_list(**args)
        else:
            result = {"error": f"Unknown tool: {fn_name}"}

        # Each tool call gets ONE tool response message, keyed by tool_call_id
        tool_responses.append(
            {
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": json.dumps(
                    result, ensure_ascii=False
                ),  # the content must be a string format.
            }
        )

    return tool_responses


In [ ]:
def chat(history, model_choice):
    print(history, model_choice)

    # Set up
    history = [
        {"role": h["role"], "content": h["content"]} for h in history
    ]  # cleaning the history by keeping all the fields that fits all model providers
    messages = [
        {"role": "system", "content": system_message}
    ] + history  # add in the system prompt and the history

    # ───── Phase 1: tool loop (NON-streaming) ─────
    while True:
        response = openai.chat.completions.create(
            model=model_choice, messages=messages, tools=tools
        )  # all the api with tools

        finish_reason = response.choices[0].finish_reason
        if finish_reason != "tool_calls":
            break  # done with tools, ready to stream the reply

        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(
            responses
        )  # extend is like the spread operator, im spreading the items in responses OUT, instead of append which is just add 1 item. responses is a list of dicts.
        print(responses)

    # ───── Phase 2: stream the final reply ─────
    stream = openai.chat.completions.create(
        model=model_choice,
        messages=messages,  # this messages is from the while loop
        tools=tools,  # keep tools available, but NO MORE SHOULD fire
        stream=True,
    )

    # Append empty assistant message — we'll grow its content as chunks arrive
    history = history + [{"role": "assistant", "content": ""}]
    accumulated = ""

    for chunk in stream:
        delta = chunk.choices[0].delta.content or ""
        if not delta:
            continue
        accumulated += delta
        history[-1]["content"] = accumulated
        yield history, None  # ← audio not ready yet

    # ───── Phase 3: TTS (text to speech) once streaming of the text is done ─────
    voice = talker(accumulated)
    yield history, voice  # ← final yield includes audio


In [ ]:
# working with the audio api
def talker(message):
    response = openai.audio.speech.create(
        model="gpt-4o-mini-tts",
        voice="onyx",  # Also, try replacing onyx with alloy or coral
        input=message,
    )
    return response.content


### Making the gradio UI


In [ ]:
# this is the function that is called when we press record


def transcribe(audio_path):
    """Whisper STT: file path → Chinese text."""
    if audio_path is None:
        return ""
    with open(audio_path, "rb") as f:
        result = openai.audio.transcriptions.create(
            model="whisper-1",
            file=f,
            language="zh",  # hint for Mandarin → big accuracy boost
            prompt="以下是普通话的句子。",  # "The following are Mandarin sentences." works 80% of the time to make sure the words are simplified not traditional chinese
        )
    return (
        result.text,
        None,
    )  # the second component is to clear the mic variable when we stop recording. without this, the mic variable will hold our recording until we press record again.


In [ ]:
# Callbacks (along with the chat() function above)

# this function is to make sure the chat bubble moves into the chat section of the ui before calling the api so it looks nicer.
def put_message_in_chatbot(message, history):
    return "", history + [{"role": "user", "content": message}]


# UI definition

# 'with' keyword is a context manager
with gr.Blocks() as ui:
    with gr.Row():
        chatbot = gr.Chatbot(height=500, type="messages")
        model_choice = gr.Dropdown(
            choices=["gpt-4.1-mini", "gpt-5.4-mini"],
            value="gpt-4.1-mini",
            label="Model",
            scale=1,
        )

    with gr.Row():
        audio_output = gr.Audio(autoplay=True)
    with gr.Row():
        message = gr.Textbox(label="Chat with our AI Assistant:")
        mic = gr.Audio(
            sources=["microphone"],
            type="filepath",  # → gives you a path to a temp WAV file
            label="🎤 Record",
            scale=1,
        )

    # When recording stops → transcribe → put text into textbox (this is an event listener, same as message.submit().then())
    mic.stop_recording(transcribe, inputs=mic, outputs=[message, mic])

    # Hooking up events to callbacks

    message.submit(
        put_message_in_chatbot, inputs=[message, chatbot], outputs=[message, chatbot]
    ).then(chat, inputs=[chatbot, model_choice], outputs=[chatbot, audio_output])

ui.launch(inbrowser=True, auth=("learner", "bananas"))
